# Candidate Generator v2.4 — Full-index rank-crowding tanısı

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YusufOck/Neural-Network-and-Genens-s-Algor-thm-Project-Erasmus-/blob/main/notebooks/candidate_generator_v2_4_rank_crowding_colab.ipynb)

Bu notebook yalnızca **N000123** çiftinin 25.000 görüntülük cervical index içindeki sırasını ve önceden belirlenmiş `K = 15, 20, 25, 30, 40, 50, 75, 100` değerlerindeki aday yükünü ölçer.

- İşlem türü: **salt-okunur tanı/audit**
- Girdi: Drive'daki tamamlanmış Candidate Generator v2.3 artifact'ları
- Çıktı: bir JSON raporu ve bir konsol logu
- Yapılmayanlar: veri ZIP'ini açma, duplicate kararı verme, family/split oluşturma, model eğitimi, v2.5 kuralı seçme

Üst menüden **Çalışma zamanı > Çalışma zamanı türünü değiştir > T4 GPU** seç. Sonra **Çalışma zamanı > Tümünü çalıştır** de. İlk hücre Google Drive izni isteyecek.

In [ ]:
# 1/4 — Drive'ı bağla, güncel kodu al ve GPU ortamını doğrula.
from google.colab import drive
from pathlib import Path
import json, subprocess, sys, torch

drive.mount('/content/drive')

REPOSITORY_URL = 'https://github.com/YusufOck/Neural-Network-and-Genens-s-Algor-thm-Project-Erasmus-.git'
PROJECT_ROOT = Path('/content/NNGA')
REQUIRED_COMMIT = 'c38c44e4cab42ddc8d2d322bebef78faa925fe8c'

if not (PROJECT_ROOT / '.git').is_dir():
    subprocess.run(['git', 'clone', '--branch', 'main', REPOSITORY_URL, str(PROJECT_ROOT)], check=True)
else:
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'checkout', 'main'], check=True)
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only', 'origin', 'main'], check=True)

CURRENT_COMMIT = subprocess.check_output(
    ['git', '-C', str(PROJECT_ROOT), 'rev-parse', 'HEAD'], text=True
).strip()
ancestor_check = subprocess.run(
    ['git', '-C', str(PROJECT_ROOT), 'merge-base', '--is-ancestor', REQUIRED_COMMIT, CURRENT_COMMIT]
)
assert ancestor_check.returncode == 0, f'Gerekli kod commit’i yok: {REQUIRED_COMMIT}'
assert torch.cuda.is_available(), 'GPU seçilmemiş. T4 GPU çalışma zamanını açıp yeniden çalıştır.'

print('Drive bağlandı.')
print('Git commit:', CURRENT_COMMIT)
print('CUDA:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# 2/4 — Drive girdilerini ve değişmez tanı sözleşmesini kontrol et.
OUTPUT_DIR = Path('/content/drive/MyDrive/NNGA_OUTPUTS/candidate_generator_v2_3_full_retrieval_v1')
REPORT_PATH = Path('/content/drive/MyDrive/NNGA_OUTPUTS/candidate_generator_v2_4_rank_crowding_v1.json')
CONFIG_PATH = PROJECT_ROOT / 'config' / 'candidate_generator_v2_4_rank_crowding.json'
ANALYZER_PATH = PROJECT_ROOT / 'src' / 'analyze_candidate_generator_v2_4_rank_crowding.py'

required_paths = {
    'v2.3 output klasörü': OUTPUT_DIR,
    'stage_manifest.json': OUTPUT_DIR / 'stage_manifest.json',
    'run_contract.json': OUTPUT_DIR / 'run_contract.json',
    'rank-crowding config': CONFIG_PATH,
    'rank-crowding analyzer': ANALYZER_PATH,
}
for label, path in required_paths.items():
    exists = path.is_dir() if label == 'v2.3 output klasörü' else path.is_file()
    print(f'{label}: {exists} — {path}')
    assert exists, f'Eksik girdi: {label} — {path}'

manifest = json.loads((OUTPUT_DIR / 'stage_manifest.json').read_text(encoding='utf-8'))
artifact_count = len(manifest.get('artifacts', {}))
print('Kayıtlı artifact:', artifact_count)
assert artifact_count == 1107, f'1.107 artifact bekleniyordu, bulunan: {artifact_count}'
print('Ön kontrol başarılı. ZIP veya dataset klasörü gerekmiyor.')

In [ ]:
# 3/4 — Salt-okunur rank-crowding tanısını çalıştır ve çıktıyı Drive loguna da yaz.
from datetime import datetime

FORCE_RERUN = False
existing_complete = False
if REPORT_PATH.is_file() and not FORCE_RERUN:
    try:
        existing = json.loads(REPORT_PATH.read_text(encoding='utf-8'))
        existing_complete = existing.get('status') == 'candidate_generator_v2_4_rank_crowding_diagnostic_complete'
    except Exception:
        existing_complete = False

if existing_complete:
    print('Tamamlanmış rapor zaten var; pahalı tanı yeniden çalıştırılmadı:')
    print(REPORT_PATH)
else:
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    LOG_PATH = REPORT_PATH.parent / f'candidate_generator_v2_4_rank_crowding_{timestamp}.log'
    command = [
        sys.executable, '-u', str(ANALYZER_PATH),
        '--output-dir', str(OUTPUT_DIR),
        '--report', str(REPORT_PATH),
        '--config', str(CONFIG_PATH),
        '--block-size', '256',
        '--device', 'cuda',
    ]
    print('Tanı başlıyor.')
    print('Komut:', ' '.join(command))
    print('Log:', LOG_PATH)
    print('Bir süre yalnız ilerleme satırları görünmesi normaldir.')

    with LOG_PATH.open('w', encoding='utf-8') as log_file:
        process = subprocess.Popen(
            command, cwd=PROJECT_ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1,
        )
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end='')
            log_file.write(line)
            log_file.flush()
        return_code = process.wait()

    print('RETURN CODE:', return_code)
    assert return_code == 0, f'Tanı tamamlanamadı. Drive logunu Codex’e gönder: {LOG_PATH}'
    assert REPORT_PATH.is_file(), 'Komut başarılı göründü fakat JSON raporu bulunamadı.'

In [ ]:
# 4/4 — Tamamlanan raporu kısa biçimde göster ve bilgisayara indir.
from google.colab import files

report = json.loads(REPORT_PATH.read_text(encoding='utf-8'))
assert report.get('status') == 'candidate_generator_v2_4_rank_crowding_diagnostic_complete'
assert report.get('does_not_modify_v2_3_artifacts') is True
assert report.get('not_a_v2_5_rule') is True

target = report['target_pair']
print('Durum:', report['status'])
print('Cihaz:', report['input']['device'])
print('Doğrulanan artifact:', report['input']['verified_artifacts_in_this_diagnostic'])
print('N000123 için gereken en küçük robust top-K:', target['minimum_required_robust_top_k'])
print('Önceden belirlenmiş K aralığında bulundu mu:', target['recovered_within_predeclared_grid'])
print('Top-15 baseline aynen yeniden üretildi mi:', report['baseline_reproduction']['all_matched'])

print('\nK projeksiyonları:')
for row in report['candidate_load_projections']:
    print(
        f"K={row['top_k']:>3} | N000123={row['target_pair_recovered']} | "
        f"toplam={row['full_counterfactual_pairs']:,} | "
        f"çift/görüntü={row['full_pairs_per_image']:.3f} | "
        f"kapasite<=25={row['passes_25_pairs_per_image_capacity']}"
    )

print('\nRapor:', REPORT_PATH)
print('Bu sonuç henüz v2.5 kuralı değildir; JSON dosyasını Codex ile birlikte inceleyeceğiz.')
files.download(str(REPORT_PATH))